In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# RAG
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import CSVLoader, WebBaseLoader
from langchain_community.document_loaders import DirectoryLoader


# vector store
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_ollama import OllamaEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

# parser
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

import json
from langchain_classic.chains.query_constructor.base import AttributeInfo

# selfquery
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline, CrossEncoderReranker
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_core.structured_query import Comparison, Comparator

# Reranking
from langchain_cohere import CohereRerank

/var/folders/v8/j43szgbd7dl3yp63myxcdyr00000gn/T/ipykernel_5603/817451159.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
# 모델 선언

llm = init_chat_model(model="gpt-6-luna", reasoning={"effort":"none"})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### 1. PDF 로드

In [4]:
loader = PyPDFLoader("./data/2018 서비스·집단 분쟁조정 사례집.pdf")
docs = loader.load()
len(docs)

200

#### 2. 첫번째 페이지 정보 출력
- page_content, metadata

In [ ]:
print(docs[0].page_content)
print(docs[0].metadata)

print(docs[11].page_content)

#### 3. 정규식을 이용한 내용 추출

In [8]:
import re

pattern = r"사\n례\s*\d+.*사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."
split_text = re.findall(pattern, "".join(docs[10].page_content))
split_text

['사\n례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.']

In [9]:
if split_text:
    parts = re.split(pattern, "".join(docs[10].page_content))
parts

['제1장\n일\n반\n분\n쟁\n조\n정\n 사\n례(\n서\n비\n스)\n제1장 일반분쟁조정 사례(서비스) ● 3\n',
 '\n세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구\n주 문\n1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n운동화, 색상 : 흰색) 1켤레를 반환한다. \n2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n을 지급한다.\n이 유\n1. 기초사실\n가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n   \n 신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n인지 단정하기 어려운바, 판단 불가하다.']

In [14]:
from pydantic import BaseModel, Field
from typing import Literal


class CaseMetaData(BaseModel):
    case_number:str = Field(description="사건번호 예:2018일나565")
    decision_date:str = Field(description="결정일자 예: 2018. 8. 7.")


pydantic_parser = PydanticOutputParser(pydantic_object=CaseMetaData)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "다음은 분쟁조정 사례에 대한 텍스트입니다. {format_instructions}"),
        ("human", "형식에 맞춰 추출해주세요 {case}")
    ]
).partial(format_instructions=pydantic_parser.get_format_instructions())

chain = prompt | llm | pydantic_parser

response = chain.invoke({'case':split_text[0]})
print(response)
print(response.case_number)
print(response.decision_date)

case_number='2018일나565' decision_date='2018. 8. 7.'
2018일나565
2018. 8. 7.


#### 주문 내용 추출

In [25]:
# 주 문 위치 찾기
re.search(r"주 문\n", parts[1]).span()

# 제목 추출
title = parts[1][:re.search(r"주 문\n", parts[1]).span()[0]].strip()
print(title)

# 내용 추출 : '주 문' 부터 시작
content = parts[1][re.search(r"주 문\n", parts[1]).span()[0]:]
print(content)

세탁 후 갑피 마모 및 경화된 가죽 
운동화에 대한 손해배상 요구
주 문
1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 
운동화, 색상 : 흰색) 1켤레를 반환한다. 
2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원
을 지급한다.
이 유
1. 기초사실
가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, 
이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. 
피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 
마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였
으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신
청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없
다는 이유로 이를 거부하였다.
나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
   
 신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만
으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제
인지 단정하기 어려운바, 판단 불가하다.


In [29]:
pdf_docs = []
case_metadatas = {}

pattern = r"사\n례\s*\d+.*사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."
for doc in docs[10:-2]:
    split_text = re.findall(pattern, "".join(doc.page_content))

    if split_text:

        # 사례번호
        case_metadatas['case_id'] = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])[0]

        parts = re.split(pattern, "".join(doc.page_content))

        if re.search(r"주 문\n", parts[1]):
            case_metadatas['title'] = parts[1][:re.search(r"주 문\n", parts[1]).span()[0]].replace("\n", "").strip()
            doc.page_content = parts[1][re.search(r"주 문\n", parts[1]).span()[0]:].strip()
        else:
            case_metadatas['title'] = ''

        # 사례번호와 결정일자 추출 후 case_metadatas
        response = chain.invoke({'case':split_text[0]})
        for k,v in dict(response).items():
            case_metadatas[k] = v

        doc.metadata.update(case_metadatas)

        pdf_docs.append(doc)

    else:
        doc.metadata.update(case_metadatas)
        pdf_docs.append(doc)

In [30]:
len(pdf_docs)

188

In [32]:
pdf_docs[1].page_content
print(pdf_docs[0].metadata)

{'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'title': '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'total_pages': 200, 'page': 10, 'page_label': '11', 'case_id': '01', 'case_number': '2018일나565', 'decision_date': '2018. 8. 7.'}


#### 분할

In [33]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [34]:
print(chunks[18].page_content[:500])

주 문
1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 
운동화, 색상 : 흰색) 1켤레를 반환한다. 
2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원
을 지급한다.
이 유
1. 기초사실
가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, 
이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. 
피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 
마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였
으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신
청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없
다는 이유로 이를 거부하였다.
나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.


In [38]:
# 문장 중간에 나오는 줄바꿈 문자는 공백 한 번으로 변경

text = re.sub(r"(?<!\.)\n"," ", chunks[18].page_content)
print(text)

주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.
이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없 다는 이유로 이를 거부하였다.
나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.


In [39]:
from copy import deepcopy

final_docs = []
for doc in chunks:
    new_doc = deepcopy(doc)

    cleaned_content = re.sub(r"(?<!\.)\n"," ", new_doc.page_content)
    new_doc.page_content = (f"### 이 사건은 `{new_doc.metadata['title']}에 대한 사례입니다.\n\n{cleaned_content}")

    final_docs.append(new_doc)


#### 벡터 저장

In [40]:
Chroma.from_documents(documents=final_docs, embedding=embeddings, persist_directory="./db/chroma_db", collection_name="consumer_cases")

In [ ]:
# 벡터 불러오기

